# LC 199 — Binary Tree Right Side View
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Trees / BFS
**Pattern:** BFS Level Snapshot — Last Node Per Level

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> The rightmost node at each
level is the one visible from the right. BFS processes
levels one at a time — just take the last node of each
level.
</div>

## Official Problem Statement

Given the `root` of a binary tree, imagine yourself
standing on the right side of it, return the values
of the nodes you can see ordered from top to bottom.

**Example 1:**
```
    1            <-  see 1
   / \
  2   3          <-  see 3
   \
    5            <-  see 5
```
```
Input:  root = [1,2,3,null,5,null,4]
Output: [1,3,4]
```
**Example 2:**
```
Input:  root = [1,null,3]
Output: [1,3]
```
**Example 3:**
```
Input:  root = []
Output: []
```

**Constraints:**
- `0 <= number of nodes <= 100`
- `-100 <= Node.val <= 100`

## What This Is Actually Asking

Stand to the right of the tree and look left.
At each floor of the tree, you only see the rightmost
node — everything behind it is hidden.
Collect those rightmost nodes, top to bottom.

## Walk Through an Example by Hand

```
Tree: [1, 2, 3, None, 5, None, 4]

      1
     / \
    2   3
     \   \
      5   4

BFS — queue starts with [1]

Level 0 — size 1:
  pop 1  -> last node = 1  -> record 1
  push 2, 3

Level 1 — size 2:
  pop 2  -> not last
  pop 3  -> last node = 3  -> record 3
  push 5 (from 2), push 4 (from 3)

Level 2 — size 2:
  pop 5  -> not last
  pop 4  -> last node = 4  -> record 4

Result: [1, 3, 4]
```

## The Picture

```
Standing on the right, looking left:

      1          <- only node at level 0 -> visible
     / \
    2   3        <- 3 is rightmost -> visible
     \   \
      5   4      <- 4 is rightmost -> visible

Eye sees:  [1, 3, 4]

BFS trick: at level_size-1 (the last pop of the
inner loop), that node is the rightmost visible one.

check:  if i == level_size - 1 -> record this value
```

## When To Use This Pattern

- When you see **"right side view"** or **"left side
  view"**, think **BFS — take last (or first) per level**
- When you need **one value per level**, think
  **level snapshot BFS, grab at position i**
- For left side view, think **same approach but
  take the FIRST node each level** (index 0)
- For DFS alternative, think **right-first traversal —
  first node seen at each new depth is the answer**

## The Approach

Use BFS with a level snapshot (same as LC 102).
For each level, track an index inside the inner loop.
When the index reaches the last position for that
level (index == level_size - 1), that node is the
rightmost — record its value.
After all levels are processed, return the collected
values.

In [ ]:
from collections import deque   # O(1) popleft for BFS
from typing import Optional, List

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(vals: list) -> Optional[TreeNode]:
    """Build tree from level-order list."""
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root


def test_harness(func):
    tests = [
        ([1,2,3,None,5,None,4], [1,3,4]),  # example 1
        ([1,None,3],            [1,3]),    # right-only
        ([],                    []),       # empty
        ([1],                   [1]),      # single node
        ([1,2,3],               [1,3]),    # full level 1
        ([1,2,None,3],          [1,2,3]),  # left-leaning
        ([1,2,3,4,5,6,7],       [1,3,7]),  # full tree
        ([1,None,2,None,3],     [1,2,3]),  # right skew
    ]

    passed = 0
    for i, (vals, expected) in enumerate(tests):
        root = build_tree(vals)
        result = func(root)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | tree={vals} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def rightSideView(
    root: Optional[TreeNode]
) -> List[int]:
    """
    Return values visible from the right side of the tree.

    BFS with level snapshot. For each level, snapshot
    size = len(queue). Process exactly size nodes.
    When the inner loop index hits size-1, that is the
    rightmost node — append its value to result.

    Time:  O(n) — every node visited once
    Space: O(w) — queue holds at most one full level
    """
    pass


# Quick debug — run this cell while building
t1 = build_tree([1,2,3,None,5,None,4])
t2 = build_tree([1,None,3])
t3 = build_tree([])
t4 = build_tree([1,2,None,3])
print(rightSideView(t1))  # [1,3,4]
print(rightSideView(t2))  # [1,3]
print(rightSideView(t3))  # []
print(rightSideView(t4))  # [1,2,3]

In [ ]:
# Uncomment and run when solution is ready
# test_harness(rightSideView)

## Complexity

| Approach | Time | Space |
|---|---|---|
| DFS right-first (depth tracking) | O(n) | O(h) |
| BFS level snapshot — last per level | O(n) | O(w) |

BFS is the natural fit — levels are explicit. DFS
requires careful right-first ordering and a depth
check, but uses less memory on wide trees.

## Real World Connection

At Citi, the server monitoring hierarchy is a tree:
root is the data center, branches are regions,
leaves are individual hosts.
The "right side view" maps to the highest-priority
host visible at each tier — the one surfaced to the
on-call dashboard when an alert fires.
Only one representative per tier is paged initially;
the others are hidden unless the first escalation
fails — exactly the level-by-level last-node pattern.
On AWS, the same view selects the last-deployed
Lambda version at each deployment stage in the
CI/CD pipeline tree.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra